# EU-Guard classifier: SFT / DPO on Kaggle (2x T4, QLoRA)

**Before the first run** (once):
1. Kaggle account with **phone verification** (needed for internet + GPU).
2. Hugging Face: accept the licenses of `CohereLabs/aya-expanse-8b` and `meta-llama/Llama-3.1-8B-Instruct`;
   create a token with **write** access.
3. This notebook: *Add-ons -> Secrets* -> add `HF_TOKEN` and attach it.
4. *Settings*: Accelerator **GPU T4 x2**, Internet **on**, Persistence: none needed.
5. *Add Input*: your uploaded dataset made by `python scripts/make_kaggle_bundle.py`.

**Running**: set the 4 values in the next cell, then *Save Version -> Save & Run All (Commit)*: it runs in the
background (up to 12 h) without the browser open. Output files (logs, report) appear under the version's *Output*.

Order: `STAGE="pilot"` once (~30 min; prints s/step -> choose MAX_TRAIN), then `STAGE="all"` repeatedly (each
session continues where the last stopped; adapters + predictions live in your private HF repos), for
`MODEL="aya8b"` and `MODEL="llama8b"`. The last session writes `results/<model>_test.md`.

In [ ]:
MODEL = "aya8b"            # aya8b | llama8b
STAGE = "pilot"            # pilot | all | score | fewshot | sft | dpo_fixed | dpo_hardneg | sftdpo_fixed | sftdpo_hardneg
HF_NAMESPACE = "CHANGE-ME" # your Hugging Face username (private repos eug-cls-* are created there)
MAX_TRAIN = 4000           # training examples per run; set from the pilot's output

In [ ]:
import glob, os, shutil, subprocess
src = os.path.dirname(glob.glob("/kaggle/input/**/pyproject.toml", recursive=True)[0])
WORK = "/kaggle/working/eu-guard"
if not os.path.exists(WORK):
    shutil.copytree(src, WORK)
os.chdir(WORK)
# Training stack pinned like pyproject.toml; keep Kaggle's CUDA torch.
subprocess.run("pip install -q transformers==5.22.2 trl==1.17.1 peft==0.21.2 datasets==5.1.0 "
               "accelerate==1.15.0 huggingface-hub==2.4.0 bitsandbytes pyyaml", shell=True, check=True)
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_NAMESPACE"] = HF_NAMESPACE
os.environ["TOKENIZERS_PARALLELISM"] = "false"
subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv; python -c 'import torch, transformers, trl, peft, bitsandbytes as b; "
               "print(torch.__version__, transformers.__version__, trl.__version__, peft.__version__, b.__version__)'",
               shell=True)

In [ ]:
import sys
cmd = [sys.executable, "scripts/kaggle_pipeline.py", "--model", MODEL, "--stage", STAGE,
       "--max-train-samples", str(MAX_TRAIN), "--hours", "11"]
with open(f"/kaggle/working/log_{MODEL}_{STAGE}.txt", "w") as log:
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end=""); log.write(line); log.flush()
assert p.wait() == 0, "pipeline failed; see the log above"
for f in glob.glob("results/*"):
    shutil.copy(f, "/kaggle/working/")